# Week 3 — Corrected Replication: Data Wrangling & Split

**Goal.** The parent paper (Gray, Shalan, Kadlec, *Sales Predictions for Video Games Using
Predictive Analytics of Market Data*, IEEE CICN 2024) predicted `Global_Sales` using, among
other features, `NA_Sales`, `EU_Sales`, `JP_Sales`, and `Other_Sales` as predictors. Since
`Global_Sales` is arithmetically the sum of those four columns, the model mostly re-learns
addition (R² = 1.000, coefficients ≈1.0 on the four regional columns and ≈0 on everything
else) — the coefficients don't tell us Genre, Platform, or Critic_Score don't matter for
sales; they tell us those variables never got a chance to matter, because the regional-sales
columns already explain 100% of the variance by construction.

**This notebook starts a corrected replication from scratch.** Same target (`Global_Sales`),
same dataset, but the four regional sales columns are excluded from the predictor set, so any
signal the model finds has to come from genuine market/reception data (genre, platform,
critic/user reception, ESRB rating, release year) rather than from an arithmetic identity.

This notebook covers the first two phases only:
1. **Data wrangling** — load, clean, and select a leakage-free feature set.
2. **Data split** — Train / Test / Validation, following the same 70/20/10 two-step split
   methodology used in `../paper_replication/data_split.ipynb`.

Modeling (Linear Regression / Decision Tree / Random Forest, matching the parent paper's
model choices) is a follow-up notebook, so the Test and Validation splits stay untouched here.

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
CSV_PATH = os.path.join("..", "paper_replication", "video_game_sales_and_ratings.csv")

pd.set_option("display.width", 120)

## 1. Load the raw dataset

Same source file as the paper replication: Kaggle
["Video Game Sales and Ratings"](https://www.kaggle.com/datasets/thedevastator/video-game-sales-and-rating).

In [2]:
df = pd.read_csv(CSV_PATH)
print(f"Raw shape: {df.shape}")
df.dtypes

Raw shape: (16719, 16)


Name                object
Platform            object
Year_of_Release    float64
Genre               object
Publisher           object
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score          object
User_Count         float64
Developer           object
Rating              object
dtype: object

In [3]:
df.isna().sum().sort_values(ascending=False)

User_Count         9129
Critic_Count       8582
Critic_Score       8582
Rating             6769
User_Score         6704
Developer          6623
Year_of_Release     269
Publisher            54
Name                  2
Genre                 2
NA_Sales              0
Platform              0
JP_Sales              0
EU_Sales              0
Other_Sales           0
Global_Sales          0
dtype: int64

## 2. Clean `User_Score`

`User_Score` is stored as text (it contains `"tbd"` for unscored titles), and it's on a
0–10 scale while `Critic_Score` is on a 0–100 scale. Coerce it to numeric and rescale it to
0–100 so the two reception scores are directly comparable as features.

In [4]:
df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce") * 10
df["User_Score"].describe()

count    7590.000000
mean       71.250461
std        15.000061
min         0.000000
25%        64.000000
50%        75.000000
75%        82.000000
max        97.000000
Name: User_Score, dtype: float64

## 3. Select a leakage-free feature set

**Target:** `Global_Sales` — same as the parent paper.

**Excluded on purpose:**
- `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales` — arithmetically sum to `Global_Sales`;
  this is the leakage the parent paper had.
- `Name` — a free-text identifier, not a generalizable feature.
- `Publisher`, `Developer` — very high cardinality with substantial missingness; excluded for
  the same reason the Week 2 score-prediction direction excluded them
  (`score_prediction_analysis/README.md`).

**Kept as predictors:** `Year_of_Release`, `Genre`, `Platform`, `Rating`, `Critic_Score`,
`Critic_Count`, `User_Score`, `User_Count` — all genuine market/reception data with no
arithmetic relationship to `Global_Sales`.

In [5]:
FEATURES = [
    "Year_of_Release",
    "Genre",
    "Platform",
    "Rating",
    "Critic_Score",
    "Critic_Count",
    "User_Score",
    "User_Count",
]
TARGET = "Global_Sales"

model_df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
model_df["Year_of_Release"] = model_df["Year_of_Release"].astype(int)

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after dropping missing values in the selected features/target: {len(model_df)}")
model_df.head()

Rows before cleaning: 16719
Rows after dropping missing values in the selected features/target: 6826


,Year_of_Release,Genre,Platform,Rating,Critic_Score,Critic_Count,User_Score,User_Count,Global_Sales
0,2006,Sports,Wii,E,76.0,51.0,80.0,322.0,82.53
1,2008,Racing,Wii,E,82.0,73.0,83.0,709.0,35.52
2,2009,Sports,Wii,E,80.0,73.0,80.0,192.0,32.77
3,2006,Platform,DS,E,89.0,65.0,85.0,431.0,29.80
4,2006,Misc,Wii,E,58.0,41.0,66.0,129.0,28.92


## 4. Sanity check — is there real signal here?

Before splitting and modeling, a quick check: do these leakage-free features actually
correlate with `Global_Sales` at all, now that they're not being crowded out by the regional
sales columns? This won't be as clean as a Pearson correlation for the categorical columns,
but it's enough to confirm there's *something* for a model to find.

In [6]:
numeric_cols = ["Year_of_Release", "Critic_Score", "Critic_Count", "User_Score", "User_Count"]
model_df[numeric_cols + [TARGET]].corr()[TARGET].sort_values(ascending=False)

Global_Sales       1.000000
Critic_Count       0.290394
User_Count         0.264133
Critic_Score       0.237582
User_Score         0.088427
Year_of_Release    0.006621
Name: Global_Sales, dtype: float64

In [7]:
for col in ["Genre", "Platform", "Rating"]:
    print(f"\nMean Global_Sales by {col}:")
    print(model_df.groupby(col)[TARGET].mean().sort_values(ascending=False).round(3).to_string())


Mean Global_Sales by Genre:
Genre
Misc            1.084
Shooter         0.945
Platform        0.937
Sports          0.884
Racing          0.820
Action          0.738
Role-Playing    0.704
Simulation      0.682
Puzzle          0.669
Fighting        0.661
Adventure       0.326
Strategy        0.260

Mean Global_Sales by Platform:
Platform
PS      1.377
Wii     1.376
PS3     1.020
PS4     1.017
X360    0.995
PS2     0.830
DS      0.824
XOne    0.815
3DS     0.795
WiiU    0.742
GBA     0.556
PSP     0.482
GC      0.458
XB      0.375
DC      0.325
PC      0.287
PSV     0.262

Mean Global_Sales by Rating:
Rating
AO      1.950
K-A     1.920
M       0.995
E       0.941
E10+    0.581
T       0.579
RP      0.030


Critic/User reception and Critic/User counts show real (if modest) positive correlation with
`Global_Sales`, and mean sales clearly differ across genre/platform/rating groups — this is
exactly the kind of signal the parent paper's coefficients couldn't show, because the
regional-sales columns had already claimed all the explanatory power. That's the gap this
replication is meant to fill in.

Note: `Genre`, `Platform`, and `Rating` are left as raw categorical columns here — one-hot
encoding happens in the modeling notebook, not here, so the split files below stay in a
human-readable, encoding-agnostic form.

## 5. Train / Test / Validation split

Same methodology as `../paper_replication/data_split.ipynb`:

- **Training set (70%)** — used to fit models.
- **Test set (20%)** — used repeatedly during development to debug and compare models.
- **Validation set (10%)** — held out and **not touched** until the models built on
  Training/Test are finalized. Used exactly once, at the end, as an unbiased check of
  generalization.

Two-step split to hit exact 70/20/10 proportions of the full cleaned dataset:
1. Carve off **Validation** (10%) first — set aside and not examined further here.
2. Split the remaining 90% into **Training** (70% of the total) and **Test** (20% of the
   total).

In [8]:
TRAIN_FRAC = 0.70
TEST_FRAC = 0.20
VAL_FRAC = 0.10
assert abs(TRAIN_FRAC + TEST_FRAC + VAL_FRAC - 1.0) < 1e-9

In [9]:
# Step 1: set aside Validation (10%) -- untouched from here on.
train_test_df, validation_df = train_test_split(
    model_df, test_size=VAL_FRAC, random_state=RANDOM_STATE
)

# Step 2: split the remaining 90% into Training (70% of total) and Test (20% of total).
relative_test_frac = TEST_FRAC / (TRAIN_FRAC + TEST_FRAC)
train_df, test_df = train_test_split(
    train_test_df, test_size=relative_test_frac, random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

## 6. Verify split sizes and proportions

In [10]:
total = len(model_df)
summary = pd.DataFrame({
    "Rows": [len(train_df), len(test_df), len(validation_df), total],
    "Proportion": [
        len(train_df) / total,
        len(test_df) / total,
        len(validation_df) / total,
        total / total,
    ],
}, index=["Training", "Test", "Validation", "Total"])
summary["Proportion"] = summary["Proportion"].map(lambda p: f"{p:.1%}")
summary

,Rows,Proportion
Training,4777,70.0%
Test,1366,20.0%
Validation,683,10.0%
Total,6826,100.0%


## 7. Save the splits and the cleaned dataset

Each split goes to its own CSV under `splits/`, so the modeling notebook can load Training
and Test freely while Validation stays untouched until the final model is chosen.

In [11]:
os.makedirs("splits", exist_ok=True)

model_df.to_csv("splits/cleaned_full.csv", index=False)
train_df.to_csv("splits/train.csv", index=False)
test_df.to_csv("splits/test.csv", index=False)
validation_df.to_csv("splits/validation.csv", index=False)

print("Saved splits/cleaned_full.csv, splits/train.csv, splits/test.csv, splits/validation.csv")

Saved splits/cleaned_full.csv, splits/train.csv, splits/test.csv, splits/validation.csv


## Next steps

With Train/Test/Validation in hand, the next notebook builds and compares models
(Linear Regression + Decision Tree, matching the parent paper's model choices, plus Random
Forest for comparison) on Training/Test, evaluated with MSE/RMSE/MAE/MAPE/R² — the same
metrics as the paper's Table II — and touches `splits/validation.csv` only once, at the end,
for the final unbiased number.